# GSLoc — main experiment notebook

Primary end-to-end example for the GSLoc place-recognition pipeline.

**What this notebook does**
1. Load graph and/or image descriptor models.
2. Run database retrieval (FAISS), optional reranking, and sequence fusion.
3. Write Recall@k reports under `data/tests/`.

Before running: set dataset roots, graph directories, list files, and checkpoint paths
(see the **Model weights** section in the repository `README.md`).


In [1]:
# Imports: Test harness, datasets, graph/image models, and plotting helpers.
from gsloc.inference.test import TestConfig, Test
from pathlib import Path
from gsloc.models import opr_graph_extention as network 
import torch
from torchvision.transforms import functional as F
from mmpr.models import MegaLoc
from gsloc.datasets import ThreeRScan, SberRobotics, ScanNet

from torchvision import transforms as T
from gsloc.utils.visual import plot_metrics_from_parquet, plot_metrics_from_experiment_dir
from gsloc.models import FoLBase
from gsloc.models import EDTformer
from gsloc.models import SelaVPRplusplusBaseRerank


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-06-18 10:31:36.332 | WARNING  | opr.optional_deps:warn_once:115 - MinkowskiEngine is not available. sparse convolutions will be disabled. See the documentation for installation instructions
xFormers not available
xFormers not available


## 1. Reproducibility

Fix random seeds for PyTorch / NumPy / Python (optional; can slow training-like kernels).


In [2]:
import random
import numpy as np

def make_deterministic(seed=0):
    """Make results deterministic. If seed == -1, do not make deterministic.
    Running the script in a deterministic way might slow it down.
    """
    if seed == -1:
        return
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
make_deterministic(0)

## 2. Evaluation & preprocess settings

Similarity modes define what counts as a true match. Sequence filters control which
frames form a query window. Image transforms must match the encoder you evaluate.


In [3]:
# Positive-pair definitions for Recall@k:
#   room  — same room (coarse)
#   pose 3 m / 180° — far / "3 m condition"
#   pose 2 m / 90°  — near / "2 m condition"
similarity_kwargs_list = [
    {
        "mode": "room",
        "trans_tol_m": 3,
        "rot_tol_deg": 180
    },
    {
        "mode": "pose",
        "trans_tol_m": 3,
        "rot_tol_deg": 180
    },
    {
        "mode": "pose",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
    },
]

# Optional filters on frames that enter a query sequence (usually "none" for base_seq_report).
seq_filter_kwargs_list = [
    {
        "seq_similarity_filter_mode": "none",
        "seq_similarity_trans_tol_m": 0.5,
        "seq_similarity_rot_tol_deg": 15
    },
    {
        "seq_similarity_filter_mode": "pose",
        "seq_similarity_trans_tol_m": 0.5,
        "seq_similarity_rot_tol_deg": 15
    },
    {
        "seq_similarity_filter_mode": "pose",
        "seq_similarity_trans_tol_m": 1,
        "seq_similarity_rot_tol_deg": 30
    },
]

# Preprocess RGB before the image encoder (resize + dataset-specific normalize / orientation).
image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Resize([322, 322], antialias=True),
    T.Lambda(lambda x: F.rotate(x, angle=-90)),  # 90° clockwise
    T.Normalize(
        mean=[0.44420420130352495, 0.41322746532289134, 0.3678658064565412], 
        std=[0.24352604373543688, 0.24045797651069503, 0.24250136992133814]
    ),
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 3. Load models

Cells below load GraphSeqLoc (GAT + MegaLoc) and image baselines used as retrieval
and/or rerank models. Comment out models you do not need for a given run.


In [4]:
# Legacy / alternative GraphSeqLoc-256 checkpoint (commented out).
# weights_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/best_model.pth")

# ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# OPR_GAT_graph_encoder = network.OPR_GATGraphEncoder(
#     in_dim=4,
#     hidden_dim=512,
#     n_layers=1,
#     num_node_classes=529, 
#     node_emb_dim=128,
#     num_edge_classes=41,
#     edge_emb_dim=128,
#     proj_dim=256,
#     edge_cont_dim=10,
#     dropout=0.1,
#     heads=4
#     ).to(device)
    
# # megaloc = torch.hub.load("gmberton/MegaLoc", "get_trained_model")
# # image_encoder = megaloc.to(device)

# graph_model256 = network.OPR_MultiModalVPRGraphEncoder(
#     graph_encoder=OPR_GAT_graph_encoder,
#     image_encoder=None,
#     image_out_dim=8448,
#     graph_out_dim=256,
#     fusion_dim=8448,
#     normalize=True,
#     graph_fusion_scale=0.05,
#     freeze_image_encoder=True,
#     mode="graph")

# missing, unexpected = graph_model256.load_state_dict(ckpt["model_state_dict"], strict=False)
# ignored_unexpected_prefixes = ("image_encoder.", "graph_encoder.convs.")
# unexpected_other = [k for k in unexpected if not k.startswith(ignored_unexpected_prefixes)]
# if unexpected_other:
#     raise RuntimeError(f"Unexpected checkpoint keys: {unexpected_other}")
# # ``missing`` includes MegaLoc hub weights and GINE conv params; those ckpt tensors appear under ``ignored_unexpected_prefixes``.

# graph_model256.to(device)
# graph_model256.eval()


In [5]:
# GraphSeqLoc: GAT graph encoder (64-d) fused with MegaLoc image branch.
# Update `weights_path` to your checkpoint (see README → Model weights).
weights_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatV3/best_model.pth")
ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)

GAT_graph_encoder = network.OPR_GATGraphEncoder64(
    in_dim=4,
    hidden_dim=512,
    n_layers=1,
    num_node_classes=529, 
    node_emb_dim=64,
    num_edge_classes=41,
    edge_emb_dim=128,
    proj_dim=64,
    edge_cont_dim=10,
    dropout=0.1,
    heads=4).to(device)

graph_model64 = network.OPR_GraphEnhancedMegaloc64(
    graph_encoder=GAT_graph_encoder,
    image_encoder=None
)

missing, unexpected = graph_model64.load_state_dict(ckpt["multimodal_state_dict"], strict=False)
ignored_unexpected_prefixes = ("image_encoder.", "graph_encoder.convs.")
unexpected_other = [k for k in unexpected if not k.startswith(ignored_unexpected_prefixes)]
if unexpected_other:
    raise RuntimeError(f"Unexpected checkpoint keys: {unexpected_other}")
# ``missing`` includes MegaLoc hub weights and GINE conv params; those ckpt tensors appear under ``ignored_unexpected_prefixes``.

graph_model64.to(device)
graph_model64.eval()


OPR_GraphEnhancedMegaloc64(
  (graph_encoder): OPR_GATGraphEncoder64(
    (edge_cont_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (edge_lbl_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (node_emb): Embedding(529, 64)
    (edge_emb): Embedding(41, 128)
    (edge_cont_mlp): Sequential(
      (0): Linear(in_features=10, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (edge_gate): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
      (3): Sigmoid()
    )
    (edge_label_proj): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (edge_fuse): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1)

### Image baselines

Weights under `weights/` by default (`FoL_base.pth`, `SelaVPRplusplus_base_rerank.pth`, …).
MegaLoc is downloaded via `torch.hub` on first use.


In [6]:
fol_base = FoLBase()  # веса из weights/FoL_base.pth
fol_base = fol_base.to(device)
fol_base.eval()

Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


FoLBase(
  (model): FoLNet(
    (backbone): DINOv2(
      (model): DinoVisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
          (norm): Identity()
        )
        (blocks): ModuleList(
          (0-11): 12 x NestedTensorBlock(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attn): MemEffAttention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=768, out_features=3072, bias=True)
              (act): GELU(approximate='none')
              (fc2): Linear(in_features=3072, out_features=768, bias=T

In [7]:
selaVPR = SelaVPRplusplusBaseRerank()  # loads weights/SelaVPRplusplus_base_rerank.pth
selaVPR = selaVPR.to(device).eval()

xFormers not available
xFormers not available
xFormers not available


In [5]:
megaLoc = MegaLoc()
megaLoc.to(device)
megaLoc.eval()

Using cache found in /home/kartashov_ga/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


MegaLoc(
  (model): MegaLocModel(
    (backbone): DINOv2(
      (model): DinoVisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
          (norm): Identity()
        )
        (blocks): ModuleList(
          (0-11): 12 x NestedTensorBlock(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attn): MemEffAttention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=768, out_features=3072, bias=True)
              (act): GELU(approximate='none')
              (fc2): Linear(in_features=3072, out_features=768, 

In [6]:
edtformer = EDTformer()
edtformer = edtformer.to(device).eval()

## 4. `run_test` helper

Wraps `gsloc.inference.test.Test`: builds caches/indexes if missing, runs inference,
saves per-frame rankings (`frames.npz`), then builds the sequence Recall report.


In [6]:
def run_test(
    tests_path, 
    dataset_path, 
    dataset_name,
    date, graph_type, 
    graphmodel_type, 
    image_model_type, 
    rerank_k, 
    per_frame_k, 
    filter_type, 
    similarity_type, 
    graph_dir, 
    edge_normalizer_path, 
    scene_list_path, 
    query_list_path,
    room_json_path,
    seq_filter_kwargs,
    similarity_kwargs,
    graph_model, 
    image_model,
    time_test,
    model_self_rerank_flag,
    rerank_descriptor_save_flag,
    scans_dir,
    ):
    """Assemble paths, build TestConfig, and run retrieval + sequence Recall.

    With `graph_model` set: first-stage graph/multimodal retrieval, then
    optional image rerank (`image_model`). With only `image_model`: image-only PR
    (and optional self-rerank via `model_self_rerank_flag`).
    Results are written under `data/tests/<date>/<dataset>/...`.
    """
Assemble paths, build TestConfig, and run retrieval + sequence Recall.\n\n    With `graph_model` set: first-stage graph/multimodal retrieval, then\n    optional image rerank (`image_model`). With only `image_model`: image-only PR\n    (and optional self-rerank via `model_self_rerank_flag`).\n    Results are written under `data/tests/<date>/<dataset>/...`.\n    """\n
    model_name = f"{graphmodel_type}x{image_model_type}" if image_model_type != "None" and graphmodel_type != "None" \
        else (graphmodel_type + "_pure") if graphmodel_type != "None" else image_model_type
    today_dataset_test_path = tests_path / date / dataset_name
    test_path = today_dataset_test_path / graph_type / model_name if graph_type != "None" else today_dataset_test_path / model_name
    index_path = today_dataset_test_path / "cache" / "indexes" / graph_type / (graphmodel_type + "graph") if graph_type != "None" else today_dataset_test_path / "cache" / "indexes" / image_model_type
    query_cache_path = today_dataset_test_path / "cache" / "query_cache" / graph_type / (graphmodel_type + "graph") if graph_type != "None" else today_dataset_test_path / "cache" / "query_cache" / image_model_type
    rerank_index_path = today_dataset_test_path / "cache" / "indexes" / image_model_type if graph_type != "None" else "None"
    rerank_query_cache_path = today_dataset_test_path / "cache" / "query_cache" / image_model_type if graph_type != "None" else "None"
    frames_path = test_path / ("rerank_k_" + str(rerank_k) + "_per_frame_k_" + str(per_frame_k)) / "frames.npz"
    bench_report_path = test_path / ("rerank_k_" + str(rerank_k) + "_per_frame_k_" + str(per_frame_k)) / filter_type / similarity_type
    modality = ["graph", "image"]

    if model_self_rerank_flag:
        # query_cache_path = query_cache_path if rerank_descriptor_save_flag else None
        rerank_index_path = str(today_dataset_test_path / "cache" / "indexes" / image_model_type) + "_rerank" if rerank_descriptor_save_flag else None
        rerank_query_cache_path = str(today_dataset_test_path / "cache" / "query_cache" / image_model_type) + "_rerank" if rerank_descriptor_save_flag else None

    if dataset_name in ["3RScan", "3RScan_BIG"]:
        dataset_class = ThreeRScan
    elif dataset_name == "SberRobotics":
        dataset_class = SberRobotics
    elif dataset_name == "ScanNet":
        dataset_class = ScanNet
    else:
        raise ValueError(f"Dataset {dataset_name} not found")

    cfg = TestConfig(
        dataset_path=dataset_path,
        test_path=test_path,
        index_path=index_path,
        rerank_index_path=rerank_index_path,
        query_cache_path=query_cache_path,
        rerank_query_cache_path=rerank_query_cache_path,
        bench_report_path=bench_report_path,
        graph_path=graph_dir,   
        dataset_class=dataset_class,
        modality=tuple(modality),
        filter_kwargs={"similarity_filter_mode": "none"},
        seq_filter_kwargs=seq_filter_kwargs,
        scene_list_path=scene_list_path,
        query_list_path=query_list_path,
        room_json_path=room_json_path,
        edge_normalizer_path=edge_normalizer_path,
        image_transform_fn=image_transform_fn,
        graph_feat_dim=4,
        graph_edge_attr_dim=10,
        graph_rotate=True,
        device=device,
        batch_size=16,
        time_test=time_test,
        num_workers=4,
        model=graph_model if graph_model is not None else image_model,
        rerank_model=image_model if graph_model is not None else None,
        model_self_rerank_flag=model_self_rerank_flag,
        rerank_descriptor_save_flag=rerank_descriptor_save_flag,
        rerank_k=rerank_k,
        per_frame_k_used=per_frame_k,
        final_k=25,
        seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
        recall_at_k=[1, 5, 10, 25],
        similarity_kwargs=similarity_kwargs,
        std_mode="global",
        scene_df_field="scene",
        pose_df_field="pose",
        frames_path=frames_path,
        scans_dir=scans_dir
    )

    test = Test(cfg)
    test.run()


## 5. Experiments

Each block is a **config cell** (paths + model choice) followed by a **run loop** over
similarity conditions (and sometimes `rerank_k`).

Naming tips:
- `graph_type`: scene-graph source (`GT`, `Makarov`, `Fross`, `Qwen`, or `None` for image-only).
- `graphmodel_type` / `image_model_type`: folder tags in `data/tests/...`.
- `filter_type`: usually `base_seq_report` (no sequence pose filter).
- `similarity_names`: `room-sim`, `pose-far-sim` (3 m), `pose-near-sim` (2 m).


### ScanNet — MegaLoc (image-only)


In [7]:
# --- Experiment config template ---
# Edit dataset/model paths and hyperparams below, then run the next cell.
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/ScanNet")
date = "26-06-18"
dataset_name = "ScanNet"
graph_type = "None" #"GT"
graphmodel_type = "None"#"64"
graph_model = None#graph_model64
image_model_type = "Megaloc"
image_model = megaLoc

rerank_k_list = [50, 100, 250, 500, 750, 1000]
rerank_k = rerank_k_list[5]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_Makarov"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/ScanNet/scans_random_200_db.txt"
query_list_path = "/mnt/external_usb_hdd/6YL/Datasets/ScanNet/scans_random_200_q.txt"
room_json_path = None
scans_dir = "scans_random_200"

time_test = True
model_self_rerank_flag = False   
rerank_descriptor_save_flag = False


In [8]:
for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[3], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model, 
            time_test=time_test,
            model_self_rerank_flag=model_self_rerank_flag,
            rerank_descriptor_save_flag=rerank_descriptor_save_flag,
            scans_dir=scans_dir
            )

2026-06-18 04:02:58.325 | INFO     | gsloc.datasets.scannet:__init__:468 - Metadata not found, rebuilding metadata for ScanNet dataset
2026-06-18 04:02:58.327 | INFO     | gsloc.datasets.scannet:build_scannet_df:312 - Scanning ScanNet dataset for 116 selected scenes...


/home/kartashov_ga/projects/GSLoc/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:2383: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)
2026-06-18 04:02:58.735 | ERROR    | gsloc.datasets.scannet:build_scannet_df:327 - Failed reading pose for /mnt/external_usb_hdd/6YL/Datasets/ScanNet/scans_random_200/scene0035_01/sens/pose/810.txt
Traceback (most recent call last):

  File "/home/kartashov_ga/.local/share/uv/python/cpython-3.10.19-linux-x86_64-gnu/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
           │         │     └ {'__name__': '__main__', '__doc__': 'Entry point for launching an IPython kernel.\n\nThis is separate from the ipykernel pack...
           │         └ <code object <module> at 0x7c70ed895630, file "/home/kartashov_ga/projects/GSLoc/.venv/lib/python3.10/site-packages/ipykernel...
           └ <function _run_code at 0x7c70ed8a49d0>
  File "/home/karta

Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-06-18/ScanNet/Megaloc/rerank_k_500_per_frame_k_25/frames.npz


Time test: 100%|██████████| 100/100 [00:04<00:00, 23.89it/s]


q_cache_dir:  /home/kartashov_ga/projects/GSLoc/data/tests/26-06-18/ScanNet/cache/query_cache/Megaloc
rq_cache_dir:  None
need_full_pr_rebuild:  True
using_unsavable_rerank:  False
need_descriptors_save:  True
need_rerank_descriptors_save:  False


Compute descriptors + PR cache: 100%|██████████| 753/753 [03:45<00:00,  3.34it/s]
2026-06-18 04:06:55.591 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 12,047 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-06-18/ScanNet/cache/query_cache/Megaloc/meta.parquet
100%|██████████| 11/11 [02:47<00:00, 15.19s/it]
2026-06-18 04:09:47.360 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-18 04:09:47.361 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-18 04:09:47.407 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-18/ScanNet/cache/indexes/Megaloc


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-06-18/ScanNet/Megaloc/rerank_k_500_per_frame_k_25/frames.npz


  0%|          | 0/11 [00:00<?, ?it/s]/home/kartashov_ga/projects/GSLoc/src/gsloc/datasets/scannet.py:731: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  pose_a = torch.as_tensor(pose_a)
100%|██████████| 11/11 [06:40<00:00, 36.43s/it]
2026-06-18 04:16:28.979 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-18 04:16:28.979 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-18 04:16:29.435 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-18/ScanNet/cache/indexes/Megal

Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-06-18/ScanNet/Megaloc/rerank_k_500_per_frame_k_25/frames.npz


100%|██████████| 11/11 [06:08<00:00, 33.53s/it]


### ScanNet — GraphSeqLoc (Makarov graphs, no image rerank)


In [7]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/ScanNet")
date = "26-06-18"
dataset_name = "ScanNet"
graph_type = "Makarov"
graphmodel_type = "64"
graph_model = graph_model64
image_model_type = "None"#"Megaloc"
image_model = None

rerank_k_list = [50, 100, 250, 500, 750, 1000]
rerank_k = rerank_k_list[5]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_Makarov_pt"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/ScanNet/scans_random_200_db.txt"
query_list_path = "/mnt/external_usb_hdd/6YL/Datasets/ScanNet/scans_random_200_q.txt"
room_json_path = None
scans_dir = "scans_random_200"

time_test = True
model_self_rerank_flag = False   
rerank_descriptor_save_flag = False

In [8]:
for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[3], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model, 
            time_test=time_test,
            model_self_rerank_flag=model_self_rerank_flag,
            rerank_descriptor_save_flag=rerank_descriptor_save_flag,
            scans_dir=scans_dir
            )

2026-06-18 10:33:38.027 | INFO     | gsloc.datasets.scannet:__init__:468 - Metadata not found, rebuilding metadata for ScanNet dataset
2026-06-18 10:33:38.030 | INFO     | gsloc.datasets.scannet:build_scannet_df:312 - Scanning ScanNet dataset for 70 selected scenes...
/home/kartashov_ga/projects/GSLoc/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:2383: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)
2026-06-18 10:33:38.679 | ERROR    | gsloc.datasets.scannet:build_scannet_df:327 - Failed reading pose for /mnt/external_usb_hdd/6YL/Datasets/ScanNet/scans_random_200/scene0035_00/sens/pose/615.txt
Traceback (most recent call last):

  File "/home/kartashov_ga/.local/share/uv/python/cpython-3.10.19-linux-x86_64-gnu/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
           │         │     └ {'__name__': '__main__', '__doc__': 'Entry point for launching an IPython kernel.\n

Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-06-18/ScanNet/Makarov/64_pure/rerank_k_500_per_frame_k_25/frames.npz


Time test: 100%|██████████| 100/100 [00:01<00:00, 93.08it/s] 


q_cache_dir:  /home/kartashov_ga/projects/GSLoc/data/tests/26-06-18/ScanNet/cache/query_cache/Makarov/64graph
rq_cache_dir:  None
need_full_pr_rebuild:  True
using_unsavable_rerank:  False
need_descriptors_save:  True
need_rerank_descriptors_save:  False


Compute descriptors + PR cache: 100%|██████████| 753/753 [00:42<00:00, 17.91it/s]
2026-06-18 10:35:50.377 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 12,047 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-06-18/ScanNet/cache/query_cache/Makarov/64graph/meta.parquet
100%|██████████| 11/11 [02:46<00:00, 15.17s/it]
2026-06-18 10:38:41.314 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-18 10:38:41.315 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-18 10:38:41.317 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-18/ScanNet/cache/indexes/Makarov/64graph


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-06-18/ScanNet/Makarov/64_pure/rerank_k_500_per_frame_k_25/frames.npz


  0%|          | 0/11 [00:00<?, ?it/s]/home/kartashov_ga/projects/GSLoc/src/gsloc/datasets/scannet.py:731: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  pose_a = torch.as_tensor(pose_a)
100%|██████████| 11/11 [03:41<00:00, 20.16s/it]
2026-06-18 10:42:23.471 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-18 10:42:23.471 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-18 10:42:23.473 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-18/ScanNet/cache/indexes/Makar

Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-06-18/ScanNet/Makarov/64_pure/rerank_k_500_per_frame_k_25/frames.npz


100%|██████████| 11/11 [03:34<00:00, 19.50s/it]


### 3RScan — GraphSeqLoc (Qwen graphs)


In [ ]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/ScanNet")
date = "26-06-01"
dataset_name = "3RScan"
graph_type = "Qwen"
graphmodel_type = "64"
graph_model = graph_model64
image_model_type = "None"#"Megaloc"
image_model = None

rerank_k_list = [50, 100, 250, 500, 750, 1000]
rerank_k = rerank_k_list[5]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_Makarov_pt"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"
query_list_path = None
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"
scans_dir = None

time_test = True
model_self_rerank_flag = False   
rerank_descriptor_save_flag = False

In [ ]:
for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[3], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model, 
            time_test=time_test,
            model_self_rerank_flag=model_self_rerank_flag,
            rerank_descriptor_save_flag=rerank_descriptor_save_flag,
            scans_dir=scans_dir
            )

### 3RScan_BIG — FoL baseline


In [26]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-05-14"
dataset_name = "3RScan_BIG"
graph_type = "None" #"GT"
graphmodel_type = "None"#"64"
graph_model = None#graph_model64
image_model_type = "Fol_base"
image_model = fol_base

rerank_k_list = [50, 100, 250, 500, 750, 1000]
rerank_k = rerank_k_list[3]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

time_test = True
model_self_rerank_flag = False   
rerank_descriptor_save_flag = False

In [27]:
for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[3], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model, 
            time_test=time_test,
            model_self_rerank_flag=model_self_rerank_flag,
            rerank_descriptor_save_flag=rerank_descriptor_save_flag
            )

2026-06-05 20:50:44.394 | INFO     | gsloc.datasets.three_rscan:__init__:353 - Metadata not found, rebuilding metadata for 3rscan dataset
2026-06-05 20:50:44.395 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:210 - Scanning 3rscan dataset for 478 selected scenes...
2026-06-05 20:51:17.363 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:278 - Scanned 50,000 frames...
2026-06-05 20:51:51.133 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:278 - Scanned 100,000 frames...
2026-06-05 20:52:13.970 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:288 - Scanned 134785 rows
2026-06-05 20:52:13.992 | INFO     | gsloc.datasets.three_rscan:__init__:353 - Metadata not found, rebuilding metadata for 3rscan dataset
2026-06-05 20:52:13.993 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:210 - Scanning 3rscan dataset for 10 selected scenes...
2026-06-05 20:52:22.214 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:288 - Scanned 2657 rows
2026-06-05 20:52:2

Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/Fol_base/rerank_k_500_per_frame_k_25/frames.npz


Time test: 100%|██████████| 100/100 [00:23<00:00,  4.21it/s]


q_cache_dir:  /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/query_cache/Fol_base
rq_cache_dir:  None
need_full_pr_rebuild:  True
using_unsavable_rerank:  False
need_descriptors_save:  True
need_rerank_descriptors_save:  False


Compute descriptors + PR cache: 100%|██████████| 167/167 [05:26<00:00,  1.96s/it]
2026-06-05 21:29:50.367 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 2,657 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/query_cache/Fol_base/meta.parquet
100%|██████████| 11/11 [00:36<00:00,  3.29s/it]
2026-06-05 21:30:28.091 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-05 21:30:28.091 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-05 21:30:28.855 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/indexes/Fol_base


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/Fol_base/rerank_k_500_per_frame_k_25/frames.npz


100%|██████████| 11/11 [01:10<00:00,  6.43s/it]
2026-06-05 21:31:43.578 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-05 21:31:43.579 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-05 21:31:44.358 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/indexes/Fol_base


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/Fol_base/rerank_k_500_per_frame_k_25/frames.npz


100%|██████████| 11/11 [01:07<00:00,  6.16s/it]


### 3RScan_BIG — GraphSeqLoc (GT) + MegaLoc rerank


In [24]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-05-14"
dataset_name = "3RScan_BIG"
graph_type = "GT"
graphmodel_type = "64"
graph_model = graph_model64
image_model_type = "Megaloc"
image_model = megaLoc

rerank_k_list = [50, 100, 250, 500, 750, 1000]
rerank_k = rerank_k_list[5]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

time_test = True
model_self_rerank_flag = False   
rerank_descriptor_save_flag = True

In [25]:
for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[5], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model, 
            time_test=time_test,
            model_self_rerank_flag=model_self_rerank_flag,
            rerank_descriptor_save_flag=rerank_descriptor_save_flag
            )

2026-06-05 20:47:01.010 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-05 20:47:01.010 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-05 20:47:01.018 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/indexes/GT/64graph
2026-06-05 20:47:01.442 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-05 20:47:01.442 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-05 20:47:02.189 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/indexes/Megaloc


Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/GT/64xMegaloc/rerank_k_1000_per_frame_k_25/frames.npz


Time test: 100%|██████████| 100/100 [00:04<00:00, 23.11it/s]


q_cache_dir:  /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/query_cache/GT/64graph
rq_cache_dir:  /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/query_cache/Megaloc
need_full_pr_rebuild:  False
using_unsavable_rerank:  False
need_descriptors_save:  False
need_rerank_descriptors_save:  False
Using cached descriptors for query: loading from  /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/query_cache/GT/64graph
Descriptors loaded:  (2657, 64) (2657, 8448) getting results


retrieval: 2657it [00:22, 117.86it/s]


...

100%|██████████| 11/11 [00:36<00:00,  3.30s/it]
2026-06-05 20:48:10.434 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-05 20:48:10.435 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-05 20:48:10.441 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/indexes/GT/64graph
2026-06-05 20:48:10.859 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-05 20:48:10.859 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-05 20:48:11.605 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/indexes/Megaloc


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/GT/64xMegaloc/rerank_k_1000_per_frame_k_25/frames.npz


100%|██████████| 11/11 [01:14<00:00,  6.75s/it]
2026-06-05 20:49:29.154 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-05 20:49:29.155 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-05 20:49:29.162 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/indexes/GT/64graph
2026-06-05 20:49:29.571 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-05 20:49:29.571 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-05 20:49:30.323 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/indexes/Megaloc


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/GT/64xMegaloc/rerank_k_1000_per_frame_k_25/frames.npz


100%|██████████| 11/11 [01:10<00:00,  6.45s/it]


### 3RScan_BIG — EDTFormer baseline


In [15]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-05-14"
dataset_name = "3RScan_BIG"
graph_type = "None" #"GT"
graphmodel_type = "None"#"64"
graph_model = None#graph_model64
image_model_type = "EDTformer"
image_model = edtformer

rerank_k_list = [50, 100, 250, 500, 750, 1000]
rerank_k = rerank_k_list[3]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

time_test = True
model_self_rerank_flag = False   
rerank_descriptor_save_flag = False

In [16]:
for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[3], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model, 
            time_test=time_test,
            model_self_rerank_flag=model_self_rerank_flag,
            rerank_descriptor_save_flag=rerank_descriptor_save_flag
            )

2026-06-06 00:08:07.087 | INFO     | gsloc.datasets.three_rscan:__init__:353 - Metadata not found, rebuilding metadata for 3rscan dataset
2026-06-06 00:08:07.088 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:210 - Scanning 3rscan dataset for 10 selected scenes...


2026-06-06 00:08:29.654 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:288 - Scanned 2657 rows
2026-06-06 00:08:29.656 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-06 00:08:29.657 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-06 00:08:30.238 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/indexes/EDTformer


Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/EDTformer/rerank_k_500_per_frame_k_25/frames.npz


Time test: 100%|██████████| 100/100 [00:13<00:00,  7.67it/s]


q_cache_dir:  /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/query_cache/EDTformer
rq_cache_dir:  None
need_full_pr_rebuild:  True
using_unsavable_rerank:  False
need_descriptors_save:  True
need_rerank_descriptors_save:  False


Compute descriptors + PR cache: 100%|██████████| 167/167 [02:52<00:00,  1.03s/it]
2026-06-06 00:11:38.179 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 2,657 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/query_cache/EDTformer/meta.parquet
100%|██████████| 11/11 [00:36<00:00,  3.33s/it]
2026-06-06 00:12:16.066 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-06 00:12:16.067 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-06 00:12:16.430 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/indexes/EDTformer


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/EDTformer/rerank_k_500_per_frame_k_25/frames.npz


100%|██████████| 11/11 [01:10<00:00,  6.42s/it]
2026-06-06 00:13:29.380 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-06 00:13:29.381 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-06 00:13:29.745 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/cache/indexes/EDTformer


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-14/3RScan_BIG/EDTformer/rerank_k_500_per_frame_k_25/frames.npz


100%|██████████| 11/11 [01:07<00:00,  6.15s/it]


### 3RScan — FoL with self-rerank


In [8]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-06-01"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "None" #"GT"
graphmodel_type = "None"#"64"
graph_model = None#graph_model64
image_model_type = "FoL_base"
image_model = fol_base

rerank_k_list = [50, 100, 250, 500, 750,1000]
rerank_k = rerank_k_list[0]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None#/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

time_test = True
model_self_rerank_flag = True   
rerank_descriptor_save_flag = False

In [ ]:
for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[0], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model, 
            time_test=time_test,
            model_self_rerank_flag=model_self_rerank_flag,
            rerank_descriptor_save_flag=rerank_descriptor_save_flag
            )

2026-06-09 15:00:26.398 | INFO     | gsloc.datasets.three_rscan:__init__:353 - Metadata not found, rebuilding metadata for 3rscan dataset
2026-06-09 15:00:26.423 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:210 - Scanning 3rscan dataset for 93 selected scenes...


### 3RScan — FoL (no self-rerank)


In [ ]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-05-19"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "None" #"GT"
graphmodel_type = "None"#"64"
graph_model = None#graph_model64
image_model_type = "Fol_base"
image_model = fol_base

rerank_k_list = [50, 100, 250, 500, 750,1000]
rerank_k = rerank_k_list[3]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None#/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

In [ ]:
for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[3], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model,   
            )

2026-05-20 22:18:41.654 | INFO     | gsloc.datasets.three_rscan:__init__:353 - Metadata not found, rebuilding metadata for 3rscan dataset
2026-05-20 22:18:41.655 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:210 - Scanning 3rscan dataset for 93 selected scenes...


2026-05-20 22:18:53.601 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:288 - Scanned 21013 rows
2026-05-20 22:18:53.606 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
100%|██████████| 591/591 [02:11<00:00,  4.49it/s]
2026-05-20 22:21:05.556 | INFO     | mmpr.inference.index:generate:466 - descriptors.npy file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/cache/indexes/Fol_base
2026-05-20 22:21:05.716 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/cache/indexes/Fol_base


Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/Fol_base/rerank_k_500_per_frame_k_25/frames.npz


Compute descriptors + PR cache: 100%|██████████| 1314/1314 [07:03<00:00,  3.10it/s]
2026-05-20 22:28:10.028 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/cache/query_cache/Fol_base/meta.parquet
100%|██████████| 11/11 [04:46<00:00, 26.04s/it]
2026-05-20 22:32:56.995 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
2026-05-20 22:32:56.995 | INFO     | mmpr.inference.index:generate:468 - Using existing descriptors.npy
2026-05-20 22:32:57.053 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/cache/indexes/Fol_base


Index search time mean: 0.09578471845841156
Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/Fol_base/rerank_k_500_per_frame_k_25/frames.npz


  0%|          | 0/11 [00:00<?, ?it/s]/home/kartashov_ga/projects/GSLoc/src/gsloc/datasets/three_rscan.py:625: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  pose_a = torch.as_tensor(pose_a)
100%|██████████| 11/11 [10:15<00:00, 55.92s/it]
2026-05-20 22:43:12.548 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
2026-05-20 22:43:12.549 | INFO     | mmpr.inference.index:generate:468 - Using existing descriptors.npy
2026-05-20 22:43:12.607 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/cache/indexes/Fo

Index search time mean: 0.0
Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/Fol_base/rerank_k_500_per_frame_k_25/frames.npz


100%|██████████| 11/11 [09:37<00:00, 52.53s/it]

Index search time mean: 0.0


### 3RScan — GraphSeqLoc (GT graphs, graph-only)


In [8]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-05-19"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "GT"
graphmodel_type = "64"
graph_model = graph_model64
image_model_type = "None"
image_model = None

rerank_k_list = [50, 100, 250, 500, 750,1000]
rerank_k = rerank_k_list[3]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None#/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

In [9]:
for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[3], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model,   
            )

TypeError: run_test() missing 1 required positional argument: 'time_test'

### 3RScan — GraphSeqLoc (GT) + MegaLoc rerank


In [8]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-06-01"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "GT"
graphmodel_type = "64"
graph_model = graph_model64
image_model_type = "Megaloc"
image_model = megaLoc

rerank_k_list = [1000]
rerank_k = rerank_k_list[0]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None#/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

time_test = True
model_self_rerank_flag = False
rerank_descriptor_save_flag = True

NameError: name 'megaLoc' is not defined

In [9]:
#for j in range(len(seq_filter_kwargs_list)):
for j in range(len(rerank_k_list)):
    for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[j], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model,   
            time_test=time_test,
            model_self_rerank_flag=model_self_rerank_flag,
            rerank_descriptor_save_flag=rerank_descriptor_save_flag
            )

NameError: name 'rerank_k_list' is not defined

### 3RScan — GraphSeqLoc (Fross) + MegaLoc rerank


In [18]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-06-01"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "Fross"
graphmodel_type = "64"
graph_model = graph_model64
image_model_type = "Megaloc"
image_model = megaLoc

rerank_k_list = [1000]
rerank_k = rerank_k_list[0]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphsFROSS_after_fix2"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

time_test = True
model_self_rerank_flag = False
rerank_descriptor_save_flag = False

In [19]:
#for j in range(len(seq_filter_kwargs_list)):
for j in range(len(rerank_k_list)):
    for i in range(0, len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path,
            dataset_path=dataset_path,
            dataset_name=dataset_name,
            date=date,
            graph_type=graph_type,
            graphmodel_type=graphmodel_type,
            image_model_type=image_model_type,
            rerank_k=rerank_k_list[j], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model,   
            time_test=time_test,
            model_self_rerank_flag=model_self_rerank_flag,
            rerank_descriptor_save_flag=rerank_descriptor_save_flag
            )

2026-06-04 00:53:58.673 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-04 00:53:58.674 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-04 00:53:58.676 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/indexes/Fross/64graph
2026-06-04 00:53:58.714 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-04 00:53:58.715 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy


2026-06-04 00:53:59.274 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/indexes/Megaloc


Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/Fross/64xMegaloc/rerank_k_1000_per_frame_k_25/frames.npz


Time test:   0%|          | 0/100 [00:00<?, ?it/s]2026-06-04 00:54:00.152 | WARNING  | gsloc.datasets.three_rscan:_warn_missing_asset:458 - Missing or unreadable graph /mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphsFROSS_after_fix2/a0905fdb-66f7-2272-9fc5-7c0008d5e87b/frame-000071.pt
2026-06-04 00:54:00.153 | WARNING  | gsloc.datasets.three_rscan:_warn_missing_asset:458 - Missing or unreadable graph /mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphsFROSS_after_fix2/2e369567-e133-204c-909a-c5da44bb58df/frame-000134.pt
2026-06-04 00:54:00.155 | WARNING  | gsloc.datasets.three_rscan:_warn_missing_asset:458 - Missing or unreadable graph /mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphsFROSS_after_fix2/42384908-60a7-271e-9c46-01e562c8974c/frame-000080.pt
2026-06-04 00:54:00.155 | WARNING  | gsloc.datasets.three_rscan:_warn_missing_asset:458 - Missing or unreadable graph /mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphsFROSS_after_fix2/c2d9933d-1947-2fbf-81fa-c8a7f9625eea/fr

q_cache_dir:  /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/query_cache/Fross/64graph
rq_cache_dir:  /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/query_cache/Megaloc
need_full_pr_rebuild:  False
using_unsavable_rerank:  False
need_descriptors_save:  False
need_rerank_descriptors_save:  False
Using cached descriptors for query: loading from  /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/query_cache/Fross/64graph
Descriptors loaded:  (21013, 64) (21013, 8448) getting results


...

100%|██████████| 11/11 [04:42<00:00, 25.70s/it]
2026-06-04 01:01:14.957 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-04 01:01:14.958 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-04 01:01:14.959 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/indexes/Fross/64graph
2026-06-04 01:01:14.985 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-04 01:01:14.986 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-04 01:01:15.454 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/indexes/Megaloc


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/Fross/64xMegaloc/rerank_k_1000_per_frame_k_25/frames.npz


100%|██████████| 11/11 [06:15<00:00, 34.13s/it]
2026-06-04 01:07:32.964 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-04 01:07:32.965 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-04 01:07:32.966 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/indexes/Fross/64graph
2026-06-04 01:07:32.991 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-04 01:07:32.992 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-04 01:07:33.242 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/indexes/Megaloc


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/Fross/64xMegaloc/rerank_k_1000_per_frame_k_25/frames.npz


100%|██████████| 11/11 [06:06<00:00, 33.34s/it]


### SberRobotics — GraphSeqLoc (Makarov) + MegaLoc


In [10]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/SberRobotics")
date = "26-06-01"
dataset_name = "Sber" #"3RScan_BIG"
graph_type = "Makarov"
graphmodel_type = "64"
graph_model = graph_model64
image_model_type = "Megaloc"
image_model = megaLoc

rerank_k_list = [1000]
rerank_k = rerank_k_list[0]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "maps/SceneGraphs_pt"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/SberRobotics/database_maps.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = "/mnt/external_usb_hdd/6YL/Datasets/SberRobotics/queries_maps.txt"
room_json_path = None

time_test = True
model_self_rerank_flag = False
rerank_descriptor_save_flag = False

In [13]:
#for j in range(len(seq_filter_kwargs_list)):
for j in range(len(rerank_k_list)):
    for i in range(1, len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[j], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model,   
            time_test=time_test,
            model_self_rerank_flag=model_self_rerank_flag,
            rerank_descriptor_save_flag=rerank_descriptor_save_flag
            )

2026-06-03 23:59:36.080 | INFO     | gsloc.datasets.sber_robotics:__init__:459 - Metadata not found, rebuilding metadata for SberRobotics dataset
2026-06-03 23:59:36.081 | INFO     | gsloc.datasets.sber_robotics:build_sber_robotics_df:312 - Scanning SberRobotics dataset for 1 selected maps...
2026-06-03 23:59:38.465 | WARNING  | gsloc.datasets.sber_robotics:iter_sber_robotics_frames:240 - Skipping map maps (missing /mnt/external_usb_hdd/6YL/Datasets/SberRobotics/maps/maps/keyframe_map/keyframe_map or poses.csv)
2026-06-03 23:59:38.470 | INFO     | gsloc.datasets.sber_robotics:build_sber_robotics_df:381 - Scanned 3258 rows
2026-06-03 23:59:38.472 | INFO     | gsloc.datasets.sber_robotics:__init__:459 - Metadata not found, rebuilding metadata for SberRobotics dataset
2026-06-03 23:59:38.472 | INFO     | gsloc.datasets.sber_robotics:build_sber_robotics_df:312 - Scanning SberRobotics dataset for 7 selected maps...


set() Index(['idx', 'scene', 'room', 'pose', 'frame_idx', 'image_path',
       'graph_path'],
      dtype='object') ('graph', 'image')


2026-06-03 23:59:40.845 | WARNING  | gsloc.datasets.sber_robotics:iter_sber_robotics_frames:240 - Skipping map maps (missing /mnt/external_usb_hdd/6YL/Datasets/SberRobotics/maps/maps/keyframe_map/keyframe_map or poses.csv)
2026-06-03 23:59:40.851 | INFO     | gsloc.datasets.sber_robotics:build_sber_robotics_df:381 - Scanned 6278 rows
2026-06-03 23:59:40.863 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 3,258 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/indexes/Makarov/64graph/meta.parquet
2026-06-03 23:59:40.864 | INFO     | mmpr.inference.index:generate:444 - meta.parquet file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/indexes/Makarov/64graph


set() Index(['idx', 'scene', 'room', 'pose', 'frame_idx', 'image_path',
       'graph_path'],
      dtype='object') ('graph', 'image')


100%|██████████| 204/204 [01:05<00:00,  3.09it/s]
2026-06-04 00:00:46.853 | INFO     | mmpr.inference.index:generate:473 - descriptors.npy file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/indexes/Makarov/64graph
2026-06-04 00:00:46.855 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/indexes/Makarov/64graph
2026-06-04 00:00:46.870 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-04 00:00:46.871 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-04 00:00:47.110 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/indexes/Megaloc


...

Time test: 100%|██████████| 100/100 [00:06<00:00, 16.58it/s]


q_cache_dir:  /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/query_cache/Makarov/64graph
rq_cache_dir:  /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/query_cache/Megaloc
need_full_pr_rebuild:  True
using_unsavable_rerank:  False
need_descriptors_save:  True
need_rerank_descriptors_save:  False


Compute descriptors + PR cache: 100%|██████████| 393/393 [03:35<00:00,  1.82it/s]
2026-06-04 00:04:29.207 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 6,278 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/query_cache/Makarov/64graph/meta.parquet
100%|██████████| 11/11 [03:47<00:00, 20.66s/it]
2026-06-04 00:08:20.440 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-04 00:08:20.441 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-04 00:08:20.442 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/indexes/Makarov/64graph
2026-06-04 00:08:20.453 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-04 00:08:20.454 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-04 00:08:20.476 | INFO     | mmpr.inference.inde

set() Index(['idx', 'scene', 'room', 'pose', 'frame_idx', 'image_path',
       'graph_path'],
      dtype='object') ('graph', 'image')
set() Index(['idx', 'scene', 'room', 'pose', 'frame_idx', 'image_path',
       'graph_path'],
      dtype='object') ('graph', 'image')
Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/Makarov/64xMegaloc/rerank_k_1000_per_frame_k_25/frames.npz


  0%|          | 0/11 [00:00<?, ?it/s]/home/kartashov_ga/projects/GSLoc/src/gsloc/datasets/sber_robotics.py:732: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  pose_a = torch.as_tensor(pose_a)
100%|██████████| 11/11 [03:13<00:00, 17.60s/it]


In [15]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/SberRobotics")
date = "26-06-01"
dataset_name = "Sber" #"3RScan_BIG"
graph_type = "None"
graphmodel_type = "None"
graph_model = None
image_model_type = "SelaVPRpp"
image_model = selaVPR

rerank_k_list = [100]
rerank_k = rerank_k_list[0]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "maps/SceneGraphs_pt"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/SberRobotics/database_maps.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = "/mnt/external_usb_hdd/6YL/Datasets/SberRobotics/queries_maps.txt"
room_json_path = None

time_test = True
model_self_rerank_flag = True
rerank_descriptor_save_flag = True

In [10]:
#for j in range(len(seq_filter_kwargs_list)):
for j in range(len(rerank_k_list)):
    for i in range(1, len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[j], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model,   
            time_test=time_test,
            model_self_rerank_flag=model_self_rerank_flag,
            rerank_descriptor_save_flag=rerank_descriptor_save_flag
            )

2026-06-03 13:55:54.501 | INFO     | gsloc.datasets.sber_robotics:__init__:459 - Metadata not found, rebuilding metadata for SberRobotics dataset
2026-06-03 13:55:54.501 | INFO     | gsloc.datasets.sber_robotics:build_sber_robotics_df:312 - Scanning SberRobotics dataset for 1 selected maps...


2026-06-03 13:55:57.188 | WARNING  | gsloc.datasets.sber_robotics:iter_sber_robotics_frames:240 - Skipping map maps (missing /mnt/external_usb_hdd/6YL/Datasets/SberRobotics/maps/maps/keyframe_map/keyframe_map or poses.csv)
2026-06-03 13:55:57.193 | INFO     | gsloc.datasets.sber_robotics:build_sber_robotics_df:381 - Scanned 3260 rows
2026-06-03 13:55:57.247 | INFO     | gsloc.datasets.sber_robotics:__init__:459 - Metadata not found, rebuilding metadata for SberRobotics dataset
2026-06-03 13:55:57.247 | INFO     | gsloc.datasets.sber_robotics:build_sber_robotics_df:312 - Scanning SberRobotics dataset for 7 selected maps...
2026-06-03 13:55:59.491 | WARNING  | gsloc.datasets.sber_robotics:iter_sber_robotics_frames:240 - Skipping map maps (missing /mnt/external_usb_hdd/6YL/Datasets/SberRobotics/maps/maps/keyframe_map/keyframe_map or poses.csv)
2026-06-03 13:55:59.498 | INFO     | gsloc.datasets.sber_robotics:build_sber_robotics_df:381 - Scanned 6284 rows
2026-06-03 13:55:59.534 | INFO    

Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/SelaVPRpp/rerank_k_100_per_frame_k_25/frames.npz


Time test: 100%|██████████| 100/100 [00:03<00:00, 26.74it/s]


q_cache_dir:  /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/query_cache/SelaVPRpp
rq_cache_dir:  /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/query_cache/SelaVPRpp_rerank
need_full_pr_rebuild:  True
using_unsavable_rerank:  False
need_descriptors_save:  True
need_rerank_descriptors_save:  True


Compute descriptors + PR cache: 100%|██████████| 393/393 [01:44<00:00,  3.75it/s]
2026-06-03 13:59:28.768 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 6,284 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/query_cache/SelaVPRpp/meta.parquet
2026-06-03 13:59:28.817 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 6,284 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/query_cache/SelaVPRpp_rerank/meta.parquet
100%|██████████| 11/11 [04:01<00:00, 21.92s/it]
2026-06-03 14:03:30.509 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-03 14:03:30.510 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-03 14:03:30.514 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/indexes/SelaVPRpp
2026-06-03 14:03:30.533 | INFO     | mmpr.inference.index:ge

Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/SelaVPRpp/rerank_k_100_per_frame_k_25/frames.npz


  0%|          | 0/11 [00:00<?, ?it/s]/home/kartashov_ga/projects/GSLoc/src/gsloc/datasets/sber_robotics.py:731: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  pose_a = torch.as_tensor(pose_a)
100%|██████████| 11/11 [03:26<00:00, 18.78s/it]


In [ ]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/SberRobotics")
date = "26-06-01"
dataset_name = "Sber" #"3RScan_BIG"
graph_type = "None"
graphmodel_type = "None"
graph_model = None
image_model_type = "Megaloc"
image_model = megaLoc

rerank_k_list = [25]
rerank_k = rerank_k_list[0]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "maps/SceneGraphs_pt"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/SberRobotics/database_maps.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = "/mnt/external_usb_hdd/6YL/Datasets/SberRobotics/queries_maps.txt"
room_json_path = None

time_test = True
model_self_rerank_flag = False
rerank_descriptor_save_flag = False

In [ ]:
#for j in range(len(seq_filter_kwargs_list)):
for j in range(len(rerank_k_list)):
    for i in range(1, len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[j], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model,   
            time_test=time_test,
            model_self_rerank_flag=model_self_rerank_flag,
            rerank_descriptor_save_flag=rerank_descriptor_save_flag
            )

2026-06-02 20:54:08.578 | INFO     | gsloc.datasets.sber_robotics:__init__:459 - Metadata not found, rebuilding metadata for SberRobotics dataset
2026-06-02 20:54:08.578 | INFO     | gsloc.datasets.sber_robotics:build_sber_robotics_df:312 - Scanning SberRobotics dataset for 7 selected maps...
2026-06-02 20:54:11.700 | WARNING  | gsloc.datasets.sber_robotics:iter_sber_robotics_frames:240 - Skipping map maps (missing /mnt/external_usb_hdd/6YL/Datasets/SberRobotics/maps/maps/keyframe_map/keyframe_map or poses.csv)
2026-06-02 20:54:11.707 | INFO     | gsloc.datasets.sber_robotics:build_sber_robotics_df:381 - Scanned 6284 rows
2026-06-02 20:54:11.709 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-02 20:54:11.710 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-02 20:54:12.111 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cac

Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/Megaloc/rerank_k_25_per_frame_k_25/frames.npz


Time test: 100%|██████████| 100/100 [00:04<00:00, 20.91it/s]


q_cache_dir:  /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/query_cache/Megaloc
rq_cache_dir:  None
need_full_pr_rebuild:  True
using_unsavable_rerank:  False
need_descriptors_save:  True
need_rerank_descriptors_save:  False


Compute descriptors + PR cache: 100%|██████████| 393/393 [02:24<00:00,  2.72it/s]
2026-06-02 20:56:42.462 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 6,284 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/cache/query_cache/Megaloc/meta.parquet
  0%|          | 0/11 [00:00<?, ?it/s]/home/kartashov_ga/projects/GSLoc/src/gsloc/datasets/sber_robotics.py:732: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  pose_b = torch.as_tensor(pose_b)
100%|██████████| 11/11 [04:04<00:00, 22.24s/it]
2026-06-02 21:00:47.273 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parq

Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/Sber/Megaloc/rerank_k_25_per_frame_k_25/frames.npz


100%|██████████| 11/11 [03:31<00:00, 19.27s/it]


### 3RScan — SelaVPR++ with self-rerank


In [11]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-06-01"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "None"
graphmodel_type = "None"
graph_model = None
image_model_type = "SelaVPRpp"
image_model = selaVPR

rerank_k_list = [100]
rerank_k = rerank_k_list[0]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None#/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

time_test = True
model_self_rerank_flag = True
rerank_descriptor_save_flag = True

In [13]:
#for j in range(len(seq_filter_kwargs_list)):
for j in range(len(rerank_k_list)):
    for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[j], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model,   
            time_test=time_test,
            model_self_rerank_flag=model_self_rerank_flag,
            rerank_descriptor_save_flag=rerank_descriptor_save_flag
            )

2026-06-02 19:34:41.778 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-02 19:34:41.790 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-02 19:34:41.879 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/indexes/SelaVPRpp
2026-06-02 19:34:41.954 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-02 19:34:41.955 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-02 19:34:42.076 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/indexes/SelaVPRpp_rerank


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/SelaVPRpp/rerank_k_100_per_frame_k_25/frames.npz


100%|██████████| 11/11 [04:42<00:00, 25.72s/it]
2026-06-02 19:39:25.355 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-02 19:39:25.356 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-02 19:39:25.360 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/indexes/SelaVPRpp
2026-06-02 19:39:25.391 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-02 19:39:25.392 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-02 19:39:25.408 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/indexes/SelaVPRpp_rerank


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/SelaVPRpp/rerank_k_100_per_frame_k_25/frames.npz


  0%|          | 0/11 [00:00<?, ?it/s]/home/kartashov_ga/projects/GSLoc/src/gsloc/datasets/three_rscan.py:632: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  pose_a = torch.as_tensor(pose_a)
100%|██████████| 11/11 [09:41<00:00, 52.83s/it]
2026-06-02 19:49:06.798 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-02 19:49:06.799 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-02 19:49:06.803 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/indexes/Se

Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/SelaVPRpp/rerank_k_100_per_frame_k_25/frames.npz


100%|██████████| 11/11 [09:08<00:00, 49.90s/it]


In [19]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-06-01"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "None"
graphmodel_type = "None"
graph_model = None
image_model_type = "FoL_base"
image_model = fol_base

rerank_k_list = [25]
rerank_k = rerank_k_list[0]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None#/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

time_test = True
model_self_rerank_flag = True
rerank_descriptor_save_flag = False

In [20]:
#for j in range(len(seq_filter_kwargs_list)):
for j in range(len(rerank_k_list)):
    for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[j], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model,   
            time_test=time_test,
            model_self_rerank_flag=model_self_rerank_flag,
            rerank_descriptor_save_flag=rerank_descriptor_save_flag
            )

2026-06-02 13:50:44.892 | INFO     | gsloc.datasets.three_rscan:__init__:353 - Metadata not found, rebuilding metadata for 3rscan dataset
2026-06-02 13:50:44.893 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:210 - Scanning 3rscan dataset for 93 selected scenes...


2026-06-02 13:52:14.989 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:288 - Scanned 21013 rows
2026-06-02 13:52:14.994 | INFO     | mmpr.inference.index:generate:446 - Using existing meta.parquet
2026-06-02 13:52:14.995 | INFO     | mmpr.inference.index:generate:475 - Using existing descriptors.npy
2026-06-02 13:52:15.054 | INFO     | mmpr.inference.index:generate:494 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/indexes/FoL_base


Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/FoL_base/rerank_k_25_per_frame_k_25/frames.npz


Time test: 100%|██████████| 100/100 [02:02<00:00,  1.22s/it]


q_cache_dir:  /home/kartashov_ga/projects/GSLoc/data/tests/26-06-01/3RScan/cache/query_cache/FoL_base
rq_cache_dir:  None
need_full_pr_rebuild:  True
using_unsavable_rerank:  True
need_descriptors_save:  True
need_rerank_descriptors_save:  False


Compute descriptors + PR cache:   3%|▎         | 43/1314 [10:15<5:03:15, 14.32s/it]


KeyboardInterrupt: 

### 3RScan — GraphSeqLoc (Makarov) + MegaLoc (paper-style run)


In [13]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-05-26"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "Makarov"
graphmodel_type = "64"
graph_model = graph_model64
image_model_type = "Megaloc"
image_model = megaLoc

rerank_k_list = [1000, 2000]
rerank_k = rerank_k_list[0]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_Makarov_FULL_TEST_pt"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None#/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

time_test = True

In [14]:
# #for j in range(len(seq_filter_kwargs_list)):
# for j in range(len(rerank_k_list)):
for i in range(len(similarity_kwargs_list)):
    run_test(
        tests_path=tests_path, 
        dataset_path=dataset_path, 
        dataset_name=dataset_name,
        date=date, 
        graph_type=graph_type, 
        graphmodel_type=graphmodel_type, 
        image_model_type=image_model_type, 
        rerank_k=rerank_k_list[0], 
        per_frame_k=per_frame_k, 
        filter_type=filter_type, 
        similarity_type=similarity_names[i], 
        graph_dir=graph_dir, 
        edge_normalizer_path=edge_normalizer_path, 
        scene_list_path=scene_list_path, 
        query_list_path=query_list_path,
        room_json_path=room_json_path, 
        seq_filter_kwargs=seq_filter_kwargs, 
        similarity_kwargs=similarity_kwargs_list[i],
        graph_model=graph_model,
        image_model=image_model,   
        time_test=time_test
        )

2026-05-26 17:19:55.002 | INFO     | gsloc.datasets.three_rscan:__init__:353 - Metadata not found, rebuilding metadata for 3rscan dataset
2026-05-26 17:19:55.046 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:210 - Scanning 3rscan dataset for 30 selected scenes...
2026-05-26 17:20:04.938 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:288 - Scanned 9449 rows
2026-05-26 17:20:04.966 | INFO     | gsloc.datasets.three_rscan:__init__:353 - Metadata not found, rebuilding metadata for 3rscan dataset
2026-05-26 17:20:04.966 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:210 - Scanning 3rscan dataset for 93 selected scenes...
2026-05-26 17:20:17.012 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:288 - Scanned 21013 rows
2026-05-26 17:20:17.041 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 9,449 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/Makarov/64graph/meta.parquet
2026-05-26 17:20:17.041 | INFO   

Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/Makarov/64xMegaloc/rerank_k_1000_per_frame_k_25/frames.npz


Compute descriptors + PR cache:  30%|███       | 399/1314 [02:13<05:01,  3.03it/s]2026-05-26 17:23:14.120 | WARNING  | gsloc.datasets.three_rscan:_warn_missing_asset:451 - Missing or unreadable graph /mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs_Makarov_FULL_TEST_pt/42384908-60a7-271e-9c46-01e562c8974c/frame-000017.pt
2026-05-26 17:23:14.132 | WARNING  | gsloc.datasets.three_rscan:_warn_missing_asset:451 - Missing or unreadable graph /mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs_Makarov_FULL_TEST_pt/42384908-60a7-271e-9c46-01e562c8974c/frame-000018.pt
Compute descriptors + PR cache: 100%|██████████| 1314/1314 [07:15<00:00,  3.02it/s]
2026-05-26 17:28:16.118 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/query_cache/Makarov/64graph/meta.parquet
2026-05-26 17:28:16.556 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/p

Index1 search time mean: 0.0019185512544813005
Rerank index2 search time mean: 0.10098108008609379


2026-05-26 17:33:24.983 | INFO     | mmpr.inference.index:generate:445 - Using existing meta.parquet
2026-05-26 17:33:24.984 | INFO     | mmpr.inference.index:generate:474 - Using existing descriptors.npy
2026-05-26 17:33:24.985 | INFO     | mmpr.inference.index:generate:493 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/Makarov/64graph
2026-05-26 17:33:25.011 | INFO     | mmpr.inference.index:generate:445 - Using existing meta.parquet
2026-05-26 17:33:25.012 | INFO     | mmpr.inference.index:generate:474 - Using existing descriptors.npy
2026-05-26 17:33:25.068 | INFO     | mmpr.inference.index:generate:493 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/Megaloc


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/Makarov/64xMegaloc/rerank_k_1000_per_frame_k_25/frames.npz


100%|██████████| 11/11 [10:34<00:00, 57.70s/it]
2026-05-26 17:44:01.215 | INFO     | mmpr.inference.index:generate:445 - Using existing meta.parquet
2026-05-26 17:44:01.216 | INFO     | mmpr.inference.index:generate:474 - Using existing descriptors.npy
2026-05-26 17:44:01.218 | INFO     | mmpr.inference.index:generate:493 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/Makarov/64graph
2026-05-26 17:44:01.244 | INFO     | mmpr.inference.index:generate:445 - Using existing meta.parquet
2026-05-26 17:44:01.244 | INFO     | mmpr.inference.index:generate:474 - Using existing descriptors.npy
2026-05-26 17:44:01.300 | INFO     | mmpr.inference.index:generate:493 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/Megaloc


Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0
Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/Makarov/64xMegaloc/rerank_k_1000_per_frame_k_25/frames.npz


100%|██████████| 11/11 [09:53<00:00, 53.96s/it]

Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0


In [ ]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-05-26"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "GT"
graphmodel_type = "64"
graph_model = graph_model64
image_model_type = "Megaloc"
image_model = megaLoc

rerank_k_list = [1000, 2000]
rerank_k = rerank_k_list[0]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None#/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

time_test = True

## 6. Quick plot of a saved report

Load `summaryresults.parquet` from a finished experiment and plot metrics.


In [5]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/Makarov/64xMegaloc/rerank_k_1000_per_frame_k_25/base_seq_report/pose-far-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/Megaloc/rerank_k_500_per_frame_k_25/base_seq_report/pose-far-sim/summaryresults.parquet",
    )

{'auc_pr': Figure({
     'data': [{'hovertemplate': 'w=%{x}<br>auc_pr=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'AAAAAAAAAAAAAAAAAAAAAAAAAAAA=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend': False,
               'type': 'scatter',
               'x': [1],
               'y': [0.0]},
              {'line': {'color': 'purple', 'dash': 'dot'},
               'mode': 'lin